# Feature mathematics and scenario proxies

- Explains the reusable calculations behind possession analysis.
- Defines a scenario as a measurable pattern over frames, not a hand-written label.
- Treats every proxy as evidence with explicit limitations.

- ATD provides normalised 2D ball and temporary-slot coordinates.
- A slot is a coordinate stream, not a guaranteed persistent player identity.
- `ball_z` is unavailable: fast 2D movement is measurable, aerial height is not.
- Smart Tagging is context, not a frame-level possession label.

## 1. Coordinate system and distance

- Convert normalised ATD coordinates to metres using `105.3 m × 68.0 m`.
- If the ball is `(b_x, b_y)` and a slot is `(p_x, p_y)`, use:

`d = sqrt(((p_x - b_x) * L)^2 + ((p_y - b_y) * W)^2)`

- Calculate this for every visible slot on every frame.
- `closest_team` is the team of the smallest-distance slot; `closest_distance_m` is that distance.
- Use it as a comfortable-control anchor only; passes, races, contests, touches, and slot swaps can mislead.

## 2. Team margin, zones, and pressure

- Let `d_H` and `d_A` be nearest Home/Away distances.
- Signed margin: `d_A - d_H`; positive favours Home, negative favours Away.
- Absolute margin measures how decisive the race is.

- Radius counts: visible H/A slots with `d <= r`.
- Counts describe support/pressure, not control by themselves.
- High opponent density makes nearest-player possession unreliable.
- Possession-zone proxy = small distance + meaningful margin + low opponent density.
- Thresholds come from configuration, not copied labels.

## 3. Time-series features

- Lagged change: `d_t - d_(t-k)`.
- Closing rate: `(d_(t-k) - d_t) / (k / FPS)`; positive means approaching.
- Rolling mean/minimum/std describe recent level and volatility.
- Persistence is the fraction of a trailing window agreeing with H or A.

- `Δx = (x_t-x_(t-1))*L`; `Δy = (y_t-y_(t-1))*W`.
- Displacement: `sqrt(Δx²+Δy²)`; speed: `displacement*FPS`.
- Direction: `atan2(Δy, Δx)`; acceleration: speed change × FPS.
- A speed drop supports reception evidence but does not prove a touch.

## 4. Missing-ball trajectory proxy

For an internal missing interval, the last observed point before the gap and the first observed point after it form a bridge. We calculate gap duration, displacement, implied speed, direction change, and the residual from a straight-line path. We also inspect nearby coordinate streams: a player whose position closes toward the bridge can support a reception hypothesis.

This is deliberately not an automatic possession fill. A short, coherent bridge between same-team anchors can be filled by a configured rule; a long or physically implausible bridge remains unresolved or `X`. The plot uses 75 frames before and after for human inspection, while the resolver uses its smaller configured trajectory context.

## 5. Reachability and reception evidence

For each team, the reachability layer estimates the fastest arrival to the current or projected ball location from player distance and configured speed assumptions. Useful reception evidence includes: a local minimum in ball-to-player distance, decreasing receiver distance, ball deceleration, agreement between nearest-team and projected receiver team, persistence after arrival, and teammate/opponent proximity.

A genuine reception should be supported after the arrival: the candidate receiver remains close for a persistence window or the team establishes a comfortable-control anchor. A one-frame touch has proximity without persistence. A switch requires a receiver from the other team plus sustained post-arrival evidence; merely touching or being closest does not force a switch.

## 6. Ball-in-flight and scenario cases

`BALL_IN_FLIGHT` is a derived 2D proxy: recent ball speed reaches the configured minimum (currently 8 m/s) and the closest player is outside the stable-control radius (currently 2 m). The decoder also excludes dead-ball/restart, reception, tracking-artifact, and miscontrol candidates. `RECEPTION_CANDIDATE` is a flight episode where arrival-time evidence is close between teams. No `z` coordinate means this includes fast ground passes and through-balls; it is not an aerial-ball detector.

The operational cases are:

- **Comfortable control:** small distance, decisive team margin, low opponent pressure, and persistence.
- **Contested ball:** both teams close or local opponent density is high; nearest-player output is treated as ambiguous.
- **Ball in flight:** fast live-ball movement without a stable controller. Keep the prior team until reception evidence confirms a change.
- **Reception candidate:** projected receiver arrival is competitive; inspect the post-arrival persistence.
- **Miscontrol / one-frame touch:** close approach followed by rapid separation without sustained control.
- **Tracking artifact / missing ball:** coordinates are absent or physically incoherent; use bounded before/after evidence, not a global fill.
- **Restart / dead ball:** Smart Tagging set-piece context, out-of-bounds geometry, or sustained whole-pitch low motion. Keep `D` through the restart regime until controlled live possession is evidenced.
- **Confirmed transition:** a later opposing-team reception anchor follows a live ambiguous interval. The interval is assigned according to the configured team-level possession convention.

## 7. Anchors, confidence, and hierarchy

Anchors are frames where evidence is strong enough to constrain the timeline. The hierarchy is configured in `config/possession_rules.yaml`: restart-control anchors first, then pressure anchors, then general control anchors. Scenario evidence without a team is diagnostic only.

Confidence is not a learned probability here. It is a transparent score assembled from evidence such as distance margin, persistence, pressure, reception signal, and restart certainty. A high score can create an anchor; a low score leaves the frame unresolved. Between two anchors, short same-team gaps can be bridged; different-team anchors require reception/transition evidence.

## 8. Modular implementation

The notebook is orchestration; calculations live in reusable functions. `build_engineered_features()` composes the frame table. `add_temporal_features()` adds lagged/rolling movement, ball kinematics, and flight evidence. The reachability, missing-ball, reception, anchor, scenario, and decoder functions add their own columns without using public labels for fitting. Models consume this common table, so a scenario or anchor family can be enabled/disabled by configuration.

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'possession' else Path.cwd()
RULES_PATH = PROJECT_ROOT / 'config' / 'possession_rules.yaml'
print('project:', PROJECT_ROOT)
print('rules:', RULES_PATH)

In [ ]:
# Optional executable demonstration. Set True when running against the local data.
RUN_FEATURE_BUILD = False
if RUN_FEATURE_BUILD:
    from possession.calculations.features import build_engineered_features
    from possession.temporal_rules import load_rules
    features = build_engineered_features(rules=load_rules(RULES_PATH))
    print(features.shape)
    display(features[['frame', 'closest_distance_m', 'team_distance_margin_m', 'ball_speed_derived_mps', 'scenario_context']].head())

## Takeaway

The system does not pretend that one nearest player solves possession. It first measures geometry and motion, then identifies the scenario in which that geometry is reliable or unreliable. Strong anchors constrain the timeline; scenario evidence explains ambiguous intervals; bounded bidirectional filling and dynamic-programming decoders enforce temporal consistency.